<a href="https://colab.research.google.com/github/RamanujSahu432/Bert-Imdb/blob/main/2305557_Bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# PART 3: TOKENIZATION GAP — BERT Sentiment Analysis (IMDb)
# Roll Numbers Ending with 2 and 7
# ============================================================
# STEP 1: Runtime > Change runtime type > T4 GPU
# STEP 2: Run this entire file
# ============================================================

# ============================================================
# CELL 1 — Install Dependencies
# ============================================================
!pip install transformers datasets tokenizers scikit-learn -q

# ============================================================
# CELL 2 — Imports
# ============================================================
import time, re
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    Trainer, TrainingArguments
)
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {device}")

# ============================================================
# CELL 3 — Load IMDb Dataset (SHUFFLED to fix label imbalance)
# ============================================================
print("\n📦 Loading IMDb dataset...")
dataset = load_dataset("imdb")

TRAIN_SIZE = 2000
TEST_SIZE  = 500
MAX_LEN    = 128

# ✅ FIX: shuffle before slicing — prevents all-same-label bug
train_data = dataset["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
test_data  = dataset["test"].shuffle(seed=42).select(range(TEST_SIZE))

train_texts  = train_data["text"]
train_labels = train_data["label"]
test_texts   = test_data["text"]
test_labels  = test_data["label"]

# Verify balanced labels
pos_train = sum(train_labels)
neg_train = len(train_labels) - pos_train
pos_test  = sum(test_labels)
neg_test  = len(test_labels) - pos_test

print(f"Train → Positive: {pos_train} | Negative: {neg_train}")
print(f"Test  → Positive: {pos_test}  | Negative: {neg_test}")
print(f"Sample: {train_texts[0][:80]}...")

# ============================================================
# CELL 4 — Shared Utilities
# ============================================================
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


def compute_metrics(eval_pred):
    preds  = np.argmax(eval_pred.predictions, axis=-1)
    labels = eval_pred.label_ids
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1"      : f1_score(labels, preds, average="weighted"),
    }


def fine_tune_bert(train_enc, test_enc, train_labels, test_labels, name):
    print(f"\n{'='*60}")
    print(f"  Training — {name}")
    print(f"{'='*60}")

    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased", num_labels=2
    )

    args = TrainingArguments(
        output_dir=f"./results_{name.replace(' ', '_')}",
        num_train_epochs=2,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy="epoch",          # ✅ fixed from evaluation_strategy
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        fp16=(device == "cuda"),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=SentimentDataset(train_enc, list(train_labels)),
        eval_dataset =SentimentDataset(test_enc,  list(test_labels)),
        compute_metrics=compute_metrics,
    )

    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0

    res  = trainer.evaluate()
    mb   = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2

    print(f"\n📊 Results [{name}]")
    print(f"   Accuracy      : {res['eval_accuracy']:.4f}")
    print(f"   F1 Score      : {res['eval_f1']:.4f}")
    print(f"   Training Time : {elapsed:.1f}s")
    print(f"   Model Size    : {mb:.2f} MB")

    return {
        "name"    : name,
        "accuracy": res["eval_accuracy"],
        "f1"      : res["eval_f1"],
        "time"    : elapsed,
        "mb"      : mb,
    }

# ============================================================
# CELL 5 — Load BERT Tokenizer
# ============================================================
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
print("✅ BERT tokenizer loaded.")

def bert_encode(texts):
    enc = bert_tokenizer(
        list(texts), truncation=True,
        padding="max_length", max_length=MAX_LEN
    )
    return {k: enc[k] for k in ["input_ids", "attention_mask", "token_type_ids"]}

# ============================================================
# CELL 6 — [1/4] BASELINE: WordPiece (BERT Default)
# ============================================================
print("\n🔤 [1/4] WordPiece Tokenization (Baseline)...")

results_wp = fine_tune_bert(
    bert_encode(train_texts),
    bert_encode(test_texts),
    train_labels, test_labels,
    "WordPiece (Baseline)"
)

# ============================================================
# CELL 7 — [2/4] BPE Tokenization
# ============================================================
print("\n🔤 [2/4] BPE Tokenization...")

bpe_tok = Tokenizer(models.BPE(unk_token="[UNK]"))
bpe_tok.pre_tokenizer = pre_tokenizers.Whitespace()
bpe_tok.train_from_iterator(
    train_texts,
    trainers.BpeTrainer(
        vocab_size=10000,
        special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]
    )
)
print(f"   BPE vocab size: {bpe_tok.get_vocab_size()}")

def bpe_encode(texts):
    ids, masks, tids = [], [], []
    for text in texts:
        joined = " ".join(bpe_tok.encode(text).tokens)
        enc = bert_tokenizer(
            joined, truncation=True,
            padding="max_length", max_length=MAX_LEN
        )
        ids.append(enc["input_ids"])
        masks.append(enc["attention_mask"])
        tids.append(enc["token_type_ids"])
    return {"input_ids": ids, "attention_mask": masks, "token_type_ids": tids}

results_bpe = fine_tune_bert(
    bpe_encode(train_texts),
    bpe_encode(test_texts),
    train_labels, test_labels,
    "BPE"
)

# ============================================================
# CELL 8 — [3/4] Character-Level Tokenization
# ============================================================
print("\n🔤 [3/4] Character-Level Tokenization...")

def char_encode(texts, char_limit=300):
    ids, masks, tids = [], [], []
    for text in texts:
        enc = bert_tokenizer(
            " ".join(list(text[:char_limit])),
            truncation=True, padding="max_length", max_length=MAX_LEN
        )
        ids.append(enc["input_ids"])
        masks.append(enc["attention_mask"])
        tids.append(enc["token_type_ids"])
    return {"input_ids": ids, "attention_mask": masks, "token_type_ids": tids}

results_char = fine_tune_bert(
    char_encode(train_texts),
    char_encode(test_texts),
    train_labels, test_labels,
    "Character-Level"
)

# ============================================================
# CELL 9 — [4/4] EXTENSION: Hybrid Tokenization (Word + Char)
# ============================================================
print("\n🔤 [4/4] Hybrid Tokenization (Word + Char fallback)...")

vocab = set()
for text in train_texts:
    vocab.update(re.findall(r"\w+", text.lower()))
print(f"   Word vocab size: {len(vocab)}")

def hybrid_tok(text, max_chars=300):
    tokens = []
    for w in re.findall(r"\w+|\S", text[:max_chars]):
        tokens += [w] if w.lower() in vocab else list(w)
    return tokens

def hybrid_encode(texts):
    ids, masks, tids = [], [], []
    for text in texts:
        enc = bert_tokenizer(
            " ".join(hybrid_tok(text)),
            truncation=True, padding="max_length", max_length=MAX_LEN
        )
        ids.append(enc["input_ids"])
        masks.append(enc["attention_mask"])
        tids.append(enc["token_type_ids"])
    return {"input_ids": ids, "attention_mask": masks, "token_type_ids": tids}

results_hybrid = fine_tune_bert(
    hybrid_encode(train_texts),
    hybrid_encode(test_texts),
    train_labels, test_labels,
    "Hybrid (Word + Char)"
)

# ============================================================
# CELL 10 — EXTENSION: Dynamic Token Merging (Demo)
# ============================================================
print("\n🔧 Dynamic Token Merging — Demo")

def dynamic_merge(tokens, min_len=2):
    merged, i = [], 0
    while i < len(tokens):
        if (i < len(tokens) - 1
                and len(tokens[i]) <= min_len
                and len(tokens[i+1]) <= min_len):
            merged.append(tokens[i] + tokens[i+1])
            i += 2
        else:
            merged.append(tokens[i])
            i += 1
    return merged

ex = "The movie was absolutely brilliant!"
print(f"\nSentence        : \"{ex}\"")
print(f"WordPiece tokens: {bert_tokenizer.tokenize(ex)}")
print(f"BPE tokens      : {bpe_tok.encode(ex).tokens}")
print(f"Char tokens     : {list(ex)}")
print(f"After Dyn Merge : {dynamic_merge(list(ex))}")

# ============================================================
# CELL 11 — Tokenization Statistics
# ============================================================
print("\n📊 Tokenization Statistics (100 samples)")
print(f"\n{'Tokenizer':<25} {'Avg Tokens':>12} {'Vocab Size':>12}")
print("-" * 52)

def tok_stats(fn, texts, name):
    toks_list = [fn(t) for t in texts[:100]]
    avg = np.mean([len(tl) for tl in toks_list])
    vsz = len({tk for tl in toks_list for tk in tl})
    print(f"{name:<25} {avg:>12.1f} {vsz:>12}")

tok_stats(lambda t: bert_tokenizer.tokenize(t),  train_texts, "WordPiece")
tok_stats(lambda t: bpe_tok.encode(t).tokens,    train_texts, "BPE")
tok_stats(lambda t: list(t),                      train_texts, "Character-Level")
tok_stats(lambda t: hybrid_tok(t),                train_texts, "Hybrid")

# ============================================================
# CELL 12 — Final Comparison Table
# ============================================================
all_results = [results_wp, results_bpe, results_char, results_hybrid]

print("\n" + "="*72)
print("  FINAL RESULTS — BERT Sentiment Analysis on IMDb (Part 3)")
print("="*72)
print(f"{'Tokenizer':<25} {'Accuracy':>10} {'F1 Score':>10} {'Time(s)':>10} {'MB':>7}")
print("-"*72)
for r in all_results:
    print(f"{r['name']:<25} {r['accuracy']:>10.4f} {r['f1']:>10.4f} "
          f"{r['time']:>10.1f} {r['mb']:>7.0f}")
print("="*72)

best    = max(all_results, key=lambda r: r["f1"])
fastest = min(all_results, key=lambda r: r["time"])

print(f"\n  🏆 Best F1      : {best['name']} (F1 = {best['f1']:.4f})")
print(f"  ⚡ Fastest      : {fastest['name']} ({fastest['time']:.1f}s)")

print("""
📌 Key Insights:
  • WordPiece — purpose-built for BERT; strong baseline (~418 MB).
  • BPE       — data-driven subword merges; handles OOV words well.
  • Char-Level— tiny vocab (~90) but very long sequences; info lost
                at MAX_LEN=128 truncation boundary → lowest accuracy.
  • Hybrid    — word-level efficiency + char fallback for OOV words;
                practical middle ground between BPE and Char-Level.
  • Dyn-Merge — reduces char sequence length, recovering some
                information lost to truncation.

  All 4 runs use bert-base-uncased (~418 MB).
  Accuracy differences are purely due to tokenization strategy.
""")

✅ Using device: cuda

📦 Loading IMDb dataset...
Train → Positive: 1000 | Negative: 1000
Test  → Positive: 246  | Negative: 254
Sample: There is no relation at all between Fortier and Profiler but the fact that both ...
✅ BERT tokenizer loaded.

🔤 [1/4] WordPiece Tokenization (Baseline)...

  Training — WordPiece (Baseline)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.434627,0.397051,0.824000,0.821780
2,0.191443,0.413495,0.852000,0.851818



📊 Results [WordPiece (Baseline)]
   Accuracy      : 0.8520
   F1 Score      : 0.8518
   Training Time : 45.5s
   Model Size    : 417.65 MB

🔤 [2/4] BPE Tokenization...
   BPE vocab size: 10000

  Training — BPE


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.541536,0.421182,0.810000,0.809841
2,0.245597,0.483573,0.826000,0.825810


Token indices sequence length is longer than the specified maximum sequence length for this model (934 > 512). Running this sequence through the model will result in indexing errors



📊 Results [Hybrid (Word + Char)]
   Accuracy      : 0.8260
   F1 Score      : 0.8258
   Training Time : 45.3s
   Model Size    : 417.65 MB

🔧 Dynamic Token Merging — Demo

Sentence        : "The movie was absolutely brilliant!"
WordPiece tokens: ['the', 'movie', 'was', 'absolutely', 'brilliant', '!']
BPE tokens      : ['The', 'movie', 'was', 'absolutely', 'brilliant', '!']
Char tokens     : ['T', 'h', 'e', ' ', 'm', 'o', 'v', 'i', 'e', ' ', 'w', 'a', 's', ' ', 'a', 'b', 's', 'o', 'l', 'u', 't', 'e', 'l', 'y', ' ', 'b', 'r', 'i', 'l', 'l', 'i', 'a', 'n', 't', '!']
After Dyn Merge : ['Th', 'e ', 'mo', 'vi', 'e ', 'wa', 's ', 'ab', 'so', 'lu', 'te', 'ly', ' b', 'ri', 'll', 'ia', 'nt', '!']

📊 Tokenization Statistics (100 samples)

Tokenizer                   Avg Tokens   Vocab Size
----------------------------------------------------
WordPiece                        347.6         5320
BPE                              351.0         5129
Character-Level                 1479.3           90
